In [1]:
#The purpose of this notebook is to optimize and XGBoost regressor on each of the 4 different feature sets, Peng, Ghorbani, Xiong and CHALPHAD.


In [1]:
#import necessary libraries
#import the required libraries
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
import re
from sklearn.model_selection import RepeatedKFold,ShuffleSplit
from CBFV import composition
from scipy.stats import sem

c:\Users\Chris\Documents\ML_Project\ML_GFA_env\Lib\site-packages\CBFV\composition.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
#Take formula column and parse it into a dataframe of element columns with atomic percentages as values. This function should be able to handle the complex formulas in the Ghorbani dataset, including nested parentheses, brackets, and braces, as well as fractions and equal splits.
def assemble_composition_df(df, formula_column):
    element_list = [
        "Ag", "Al", "Am", "As", "Au",
        "B", "Ba", "Be", "Bi",
        "C", "Ca", "Cd", "Ce", "Co", "Cr", "Cs", "Cu",
        "Dy",
        "Er", "Eu",
        "Fe",
        "Ga", "Gd", "Ge",
        "H", "Hf", "Hg", "Ho",
        "In", "Ir",
        "K",
        "La", "Li", "Lu",
        "Mg", "Mn", "Mo",
        "N", "Na", "Nb", "Nd", "Ni", "Np",
        "O", "Os",
        "P", "Pa", "Pb", "Pd", "Pr", "Pt", "Pu",
        "Rb", "Re", "Rh", "Ru",
        "S", "Sb", "Sc", "Se", "Si", "Sm", "Sn", "Sr",
        "Ta", "Tb", "Tc", "Te", "Th", "Ti", "Tl", "Tm",
        "U",
        "V",
        "W",
        "Y", "Yb",
        "Zn", "Zr"
    ]
    
    def parse_fraction(s):
        """Parse a string that might be a fraction (e.g., '5/6') or a number."""
        if '/' in s:
            num, denom = s.split('/')
            return float(num) / float(denom)
        return float(s)
    
    def parse_element_composition(formula_str):
        """
        Parse element-number pairs from a formula string.
        Returns a dict of {element: amount}
        """
        composition = {}
        # Pattern to match element followed by optional number (including fractions)
        pattern = r'([A-Z][a-z]?)(\d+(?:\.\d+)?(?:/\d+(?:\.\d+)?)?)?'
        
        matches = re.findall(pattern, formula_str)
        for element, amount in matches:
            if element and element in element_list:
                if amount:
                    val = parse_fraction(amount)
                else:
                    val = 1.0
                composition[element] = composition.get(element, 0) + val
        
        return composition
    
    def parse_formula(formula):
        """
        Parse a complete alloy formula handling nested brackets, parentheses, and braces.
        Returns a dict of {element: atomic_percent}
        """
        composition = {}
        
        # Remove citation references like [24], [30], etc. at the end
        formula = re.sub(r'\[\d+\]$', '', formula)
        formula = re.sub(r'\[\d+\]', '', formula)
        
        # Remove spaces and commas used as separators
        formula = formula.replace(' ', '').replace(',', '')
        
        def process_innermost_group(f):
            """Find and process the innermost bracketed group."""
            pattern = r'([\(\[\{])([^\(\)\[\]\{\}]+)([\)\]\}])(\d+(?:\.\d+)?)?'
            
            match = re.search(pattern, f)
            if not match:
                return f, False
            
            open_bracket, content, close_bracket, multiplier = match.groups()
            
            # Parse the content of the group
            inner_comp = parse_element_composition(content)
            
            # Calculate the sum of inner compositions
            inner_sum = sum(inner_comp.values())
            
            # Determine the multiplier
            if multiplier:
                mult = float(multiplier)
            else:
                mult = 1.0
            
            # Determine if inner values are fractions or percentages
            # If sum is close to 1, treat as fractions; if close to 100, treat as percentages
            if inner_sum > 1.5:  # Likely percentages within the group
                # Normalize to fractions, then multiply
                inner_comp = {k: v / inner_sum for k, v in inner_comp.items()}
            
            # Apply multiplier
            expanded = {elem: amt * mult for elem, amt in inner_comp.items()}
            
            # Create replacement string
            replacement_parts = []
            for elem, amt in expanded.items():
                replacement_parts.append(f"{elem}{amt}")
            replacement = ''.join(replacement_parts)
            
            new_f = f[:match.start()] + replacement + f[match.end():]
            
            return new_f, True
        
        # Iteratively process innermost groups until none remain
        processed_formula = formula
        max_iterations = 20
        iteration = 0
        
        while iteration < max_iterations:
            processed_formula, found = process_innermost_group(processed_formula)
            if not found:
                break
            iteration += 1
        
        # Now parse the final expanded formula
        composition = parse_element_composition(processed_formula)
        
        # Handle equal split case (elements with no numbers)
        total = sum(composition.values())
        num_elements = len(composition)
        
        # Check if all elements have value 1.0 (no numbers given)
        if num_elements > 0 and all(v == 1.0 for v in composition.values()):
            equal_share = 100.0 / num_elements
            composition = {k: equal_share for k in composition}
        # If total is very small (< 2), scale up to 100
        elif total > 0 and total < 2:
            scale = 100.0 / total
            composition = {k: v * scale for k, v in composition.items()}
        
        return composition
    
    # Process all formulas
    composition_dicts = []
    for formula in df[formula_column]:
        try:
            comp = parse_formula(str(formula))
            composition_dicts.append(comp)
        except Exception as e:
            print(f"Error parsing '{formula}': {e}")
            composition_dicts.append({})
    
    # Create DataFrame with element columns
    comp_df = pd.DataFrame(composition_dicts)
    
    # Ensure all element columns exist, fill missing with 0
    for elem in element_list:
        if elem not in comp_df.columns:
            comp_df[elem] = 0.0
    
    # Reorder columns to match element_list and fill NaN with 0
    comp_df = comp_df.reindex(columns=element_list, fill_value=0.0)
    comp_df = comp_df.fillna(0.0)
    
    return comp_df

#take composition df and produce composition strings
def canonical_comp_string(df, tol=1e-9, decimals=2):
    element_cols = sorted([c for c in df.columns if c != "Composition String"])
    out = []
    for _, row in df[element_cols].iterrows():
        vals = row.astype(float).fillna(0.0).to_numpy()
        vals[vals < tol] = 0.0
        s = vals.sum()
        if s <= 0:
            out.append("")
            continue
        vals = vals / s * 100.0
        vals = np.round(vals, decimals)
        parts = [f"{el}{v:.{decimals}f}" for el, v in zip(element_cols, vals) if v > 0]
        out.append("".join(parts))
    return out

#create function that takes a composition df and gets the index of rows who sum to greater than 100. This is to catch any errors in the composition parsing where the percentages add up to more than 100.
def find_rows_sum_greater_than_100(df):
    over_100_index = df.sum(axis=1) > 100
    return over_100_index



In [20]:
#load the three paper datasets

raw_Xiong_df = pd.read_excel(r"Data\Paper Data\XIONG 2021.xlsx")
raw_Peng_df = pd.read_csv(r"Data\Paper Data\PENG 2021.csv")

raw_Peng_df.head()

,Unnamed: 0,Elemental components,Tg /K,Tx /K,Tl /K,Dmax /mm
0,1,Ag30.8Ca30.7Mg23.1Cu15.4,413.0,432,803,2.5
1,2,Ag30.8Mg30.8Ca30.7Cu7.7,407.0,427,809,2.0
2,3,Ag38.4Mg30.8Ca30.8,394.0,426,805,0.5
3,4,Ag38.4Mg38.4Ca23.2,391.0,425,796,1.1
4,5,Ag38.5Ca30.8Mg23Cu7.7,384.0,416,854,2.0


In [18]:
#load and process the Ghorbani dataset 
raw_Ghorbani_df = pd.read_excel(r"Data\Paper Data\Ghorbani, 2022.xlsx")

#Convert the alloys in Ghorbani dataset into cananical composition strings
Ghorbani_composition_df = assemble_composition_df(raw_Ghorbani_df, "Alloy")
Ghorbani_canonical_strings = canonical_comp_string(Ghorbani_composition_df)

#save the original length of the Ghorbani dataset for later comparison after dropping rows that sum to greater than 100
original_Ghorbani_length = len(raw_Ghorbani_df)

#replace the Ghorbani alloy column with the canonical composition strings and rename as composition string
raw_Ghorbani_df["Composition String"] = Ghorbani_canonical_strings
raw_Ghorbani_df = raw_Ghorbani_df.drop(columns=["Alloy"])

#get the index of any rows in the composition df that sum to greater than 100
Ghorbani_over_100_index = find_rows_sum_greater_than_100(Ghorbani_composition_df)
print(f"Number of rows in Ghorbani composition df that sum to greater than 100: {Ghorbani_over_100_index.sum()}")

#drop the rows over 100 from the composition df and the original df
raw_Ghorbani_df = raw_Ghorbani_df[~Ghorbani_over_100_index].reset_index(drop=True)
Ghorbani_composition_df = Ghorbani_composition_df[~Ghorbani_over_100_index].reset_index(drop=True)

#Check for any duplicate composition strings in the Ghorbani dataset and replace with the mean of the duplicates
raw_Ghorbani_df = raw_Ghorbani_df.groupby("Composition String").mean().reset_index()

#calculate the post processing length of the Ghorbani dataset and print the number of rows dropped
post_processing_Ghorbani_length = len(raw_Ghorbani_df)

#cacluate the number of duplicate rows in the Ghorbani dataset and print
ghorbani_duplicate_count = original_Ghorbani_length - post_processing_Ghorbani_length - Ghorbani_over_100_index.sum()
print(f"Number of duplicate rows in Ghorbani dataset that were averaged: {ghorbani_duplicate_count}")
print(f"Number of rows dropped from Ghorbani dataset after processing: {original_Ghorbani_length - post_processing_Ghorbani_length}")

#create a set of the Ghorbani composition strings for later comparison with the other datasets
Ghorbani_composition_strings_set = set(raw_Ghorbani_df["Composition String"])

Number of rows in Ghorbani composition df that sum to greater than 100: 26
Number of duplicate rows in Ghorbani dataset that were averaged: 28
Number of rows dropped from Ghorbani dataset after processing: 54
